In [1]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [2]:
train_data= pd.read_csv("../Data/samsum-train.csv")
val_data = pd.read_csv("../Data/samsum-validation.csv")

### Data- Preprocessing

In [3]:
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ",text)
    text = re.sub(r"\s+"," ",text)
    text = re.sub(r"<.*?>"," ",text)
    text.strip().lower()
    return text 

In [4]:
train_data["dialogue"] = train_data["dialogue"].fillna("").astype(str).apply(clean_data)
train_data["summary"] = train_data["summary"].fillna("").astype(str).apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].fillna("").astype(str).apply(clean_data)
val_data["summary"] = val_data["summary"].fillna("").astype(str).apply(clean_data)

In [5]:
train_data["dialogue"][0]

"Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)"

### Tokenize

In [6]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
# raw data =>tokenize 

def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets = tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [8]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()


### Working with the Model

In [9]:
import torch 

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(device)

cuda


In [10]:
#NLP =>Generation task


model = T5ForConditionalGeneration.from_pretrained("t5-small")
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [11]:
training_args = TrainingArguments(
    output_dir="./results",

    # Core (keep stable)
    num_train_epochs=4,  
    weight_decay=0.01,

    per_device_train_batch_size=8,   
    per_device_eval_batch_size=8,

    

   
    evaluation_strategy="epoch",
    save_strategy="epoch",

    
    logging_steps=500,

    
    warmup_steps=200,

    
    fp16=True,                     
    dataloader_num_workers=0,    

    
    save_total_limit=2,
    report_to="none"
)

d:\Vs code\AIML\.venv\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [12]:
trainer = Trainer(
    model=model,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

d:\Vs code\AIML\.venv\Lib\site-packages\accelerate\accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [13]:
#Train the module
trainer.train()

  0%|          | 0/7368 [00:00<?, ?it/s]

{'loss': 2.9916, 'grad_norm': 0.6171715259552002, 'learning_rate': 4.7949218750000005e-05, 'epoch': 0.27}
{'loss': 0.4156, 'grad_norm': 0.661960244178772, 'learning_rate': 4.446149553571429e-05, 'epoch': 0.54}
{'loss': 0.4005, 'grad_norm': 0.4963187277317047, 'learning_rate': 4.0973772321428574e-05, 'epoch': 0.81}


  0%|          | 0/103 [00:00<?, ?it/s]

{'eval_loss': 0.34791940450668335, 'eval_runtime': 8.9384, 'eval_samples_per_second': 91.515, 'eval_steps_per_second': 11.523, 'epoch': 1.0}
{'loss': 0.3844, 'grad_norm': 0.5436995029449463, 'learning_rate': 3.748604910714286e-05, 'epoch': 1.09}
{'loss': 0.3781, 'grad_norm': 0.5094355344772339, 'learning_rate': 3.399832589285715e-05, 'epoch': 1.36}
{'loss': 0.3798, 'grad_norm': 0.7014603614807129, 'learning_rate': 3.051060267857143e-05, 'epoch': 1.63}
{'loss': 0.3683, 'grad_norm': 0.7640841603279114, 'learning_rate': 2.7022879464285715e-05, 'epoch': 1.9}


  0%|          | 0/103 [00:00<?, ?it/s]

...

  0%|          | 0/103 [00:00<?, ?it/s]

{'eval_loss': 0.33290016651153564, 'eval_runtime': 6.4245, 'eval_samples_per_second': 127.324, 'eval_steps_per_second': 16.032, 'epoch': 3.0}
{'loss': 0.359, 'grad_norm': 0.44199851155281067, 'learning_rate': 9.584263392857143e-06, 'epoch': 3.26}
{'loss': 0.3573, 'grad_norm': 0.5671660304069519, 'learning_rate': 6.0965401785714294e-06, 'epoch': 3.53}
{'loss': 0.3533, 'grad_norm': 0.523043692111969, 'learning_rate': 2.6088169642857144e-06, 'epoch': 3.8}


  0%|          | 0/103 [00:00<?, ?it/s]

{'eval_loss': 0.33215832710266113, 'eval_runtime': 6.4141, 'eval_samples_per_second': 127.531, 'eval_steps_per_second': 16.058, 'epoch': 4.0}
{'train_runtime': 2182.5134, 'train_samples_per_second': 27.0, 'train_steps_per_second': 3.376, 'train_loss': 0.54980156450137, 'epoch': 4.0}


TrainOutput(global_step=7368, training_loss=0.54980156450137, metrics={'train_runtime': 2182.5134, 'train_samples_per_second': 27.0, 'train_steps_per_second': 3.376, 'total_flos': 7975421677142016.0, 'train_loss': 0.54980156450137, 'epoch': 4.0})

In [14]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\special_tokens_map.json',
 './saved_summary_model\\spiece.model',
 './saved_summary_model\\added_tokens.json')

In [15]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### Test the core logic for summarization

In [16]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) #clean

    #tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"
    ).to(device)

    #generate the summary =>token ids
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams= 5,
        early_stopping = True
    )


    #Token ids convert to summary => decoding
    summary = tokenizer.decode(targets[0],skip_special_tokens=True)
    return summary

In [17]:
test_dialogue = """Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)"""

summary = summarize_dialogue(test_dialogue)
print("Summary:" ,summary)

Summary: Rob likes the train part. Rob will watch some of his stand-ups on youtube.
